In [1]:
import pandas as pd
import gc
import numpy as np

In [2]:
df3 = pd.read_pickle('../data/df3.pkl')

In [3]:
df3.info()
df3.shape # (3846784, 30)
df3.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3846784 entries, 0 to 3846783
Data columns (total 30 columns):
 #   Column              Dtype         
---  ------              -----         
 0   BUS_SVC_NUM         float64       
 1   CRD_NUM             int64         
 2   DEST_LOC_ID_NUM     int64         
 3   ENTRY_DT            datetime64[s] 
 4   ENTRY_TM            datetime64[ns]
 5   EXIT_DT             datetime64[s] 
 6   EXIT_TM             datetime64[ns]
 7   JRNY_ID_NUM         int64         
 8   ORIG_LOC_ID_NUM     int64         
 9   RIDE_DISC_AMT       float64       
 10  RIDE_DIST_KM_CNT    float64       
 11  RIDE_FARE_AMT       float64       
 12  RIDE_ID_NUM         int64         
 13  RIDE_MIN_CNT        float64       
 14  PATRON_CATG_ID_NUM  int64         
 15  TRNSPT_MODE_CD      int64         
 16  DEST_STATION_NAME   object        
 17  DEST_MRK_ID_NUM     float64       
 18  DEST_LATITUDE       float64       
 19  DEST_LONGITUDE      float64       
 20  DE

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance
0,NaN,100005879220,22,2025-02-11,2025-02-11 11:20:34,2025-02-11,2025-02-11 11:32:25,110710909992,44,0.00,...,TRAIN,Admiralty,44.0,1.440589,103.800990,TRAIN,1.429443,103.835005,Yishun,0.0
1,NaN,100005879220,44,2025-02-11,2025-02-11 22:32:59,2025-02-11,2025-02-11 22:45:03,110710523649,22,0.00,...,TRAIN,Yishun,22.0,1.429443,103.835005,TRAIN,NaN,NaN,NaN,NaN
2,45.0,100006223599,3968,2025-02-11,2025-02-11 06:56:13,2025-02-11,2025-02-11 07:07:33,110710081977,4009,0.00,...,BUS,Blk 172,4009.0,1.350814,103.889844,BUS,1.349708,103.873575,Serangoon NEL,195.0
3,NaN,100006223599,109,2025-02-11,2025-02-11 07:09:17,2025-02-11,2025-02-11 07:18:09,110710081977,106,0.16,...,TRAIN,Serangoon NEL,106.0,1.349708,103.873575,TRAIN,1.319336,103.861570,Boon Keng,0.0
4,NaN,100006223599,106,2025-02-11,2025-02-11 14:51:22,2025-02-11,2025-02-11 15:03:53,110710904744,109,0.00,...,TRAIN,Boon Keng,109.0,1.319336,103.861570,TRAIN,1.348979,103.872774,S'Goon Stn Exit B,139.0
5,45.0,100006223599,4008,2025-02-11,2025-02-11 15:10:02,2025-02-11,2025-02-11 15:21:55,110710226201,3967,0.00,...,BUS,S'Goon Stn Exit B,3967.0,1.348979,103.872774,BUS,NaN,NaN,NaN,NaN
6,382.0,130013244516,5960,2025-02-11,2025-02-11 10:07:29,2025-02-11,2025-02-11 10:11:38,110710535706,5953,0.00,...,BUS,Blk 322 CP,5953.0,1.410690,103.897592,BUS,1.408195,103.905669,One Punggol,123.0
7,382.0,130013244516,5954,2025-02-11,2025-02-11 12:15:49,2025-02-11,2025-02-11 12:21:50,110709913132,5959,0.00,...,BUS,One Punggol,5959.0,1.408195,103.905669,BUS,NaN,NaN,NaN,NaN
8,871.0,130013244638,2782,2025-02-11,2025-02-11 06:40:58,2025-02-11,2025-02-11 06:54:37,110711365561,6011,0.00,...,BUS,Tengah CC,6011.0,1.356473,103.734424,BUS,1.358612,103.751791,Bukit Gombak,248.0
9,NaN,130013244638,45,2025-02-11,2025-02-11 06:58:38,2025-02-11,2025-02-11 07:20:00,110711365561,29,0.50,...,TRAIN,Bukit Gombak,29.0,1.358612,103.751791,TRAIN,1.436946,103.785936,Woodlands Int B5,25.0


### Information on the Patron Categories: 

- Student (Primary 1 to JC/Poly) --> 7 - 19 years old
- Adult --> 20 - 59 years old
- Senior Citizen --> 60 years and above

Average Walking Speeds by Age:
- Children (6-12 years): Around 3.0 to 4.5 km/h (1.9 to 2.8 mph)
- Adolescents (13-19 years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)
- Adults (20-59 years): Around 5.0 to 6.0 km/h (3.1 to 3.7 mph)
- Older Adults (60+ years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)

To calculate the walking speed of each group, I take the midpoint of the range. 
For Children and Adolescents, I took the midpoint of both ranges and took the average.

In [4]:
# We first filter for 3 relevant PATRON_CATG_ID_NUM.
patron_map = {1: 'Adult', 3: 'Student', 4: 'Senior Citizen'}
df3['PATRON_CATG_DESC_TXT'] = df3['PATRON_CATG_ID_NUM'].map(patron_map)

walking_speed_ms = {
    'Student': 4.125 / 3.6,      # (3.75 + 4.5) / 2 = 4.125 km/h → ~1.146 m/s
    'Adult': 5.5 / 3.6,          # ~1.528 m/s
    'Senior Citizen': 4.5 / 3.6  # 1.25 m/s
}

df3['walking_speed_ms'] = df3['PATRON_CATG_DESC_TXT'].map(walking_speed_ms)

In [ ]:
df3.head(20)

# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: df3 is already sorted by CRD_NUM and ENTRY_TM. Flag last row of each CRD_NUM
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


In [5]:
df3 = df3.sort_values(["CRD_NUM", "ENTRY_TM"]).reset_index(drop=True)

In [6]:
df3["next_ENTRY_TM"] = df3.groupby("CRD_NUM")["ENTRY_TM"].shift(-1)
df3["next_ORIG_LOC_ID_NUM"] = df3.groupby("CRD_NUM")["ORIG_LOC_ID_NUM"].shift(-1)
df3["next_BUS_SVC_NUM"] = df3.groupby("CRD_NUM")["BUS_SVC_NUM"].shift(-1)
df3["next_TRNSPT_MODE_CD"] = df3.groupby("CRD_NUM")["TRNSPT_MODE_CD"].shift(-1)

In [ ]:
df3.head(20)

In [8]:
df3["is_last_stage"] = df3["next_ENTRY_TM"].isna()

df3['missing_info'] = (
    df3['ENTRY_TM'].isna() |
    df3['EXIT_TM'].isna() |
    df3['ORIG_LATITUDE'].isna() |
    df3['ORIG_LONGITUDE'].isna() |
    df3['DEST_LATITUDE'].isna() |
    df3['DEST_LONGITUDE'].isna() |
    df3["ORIG_LOC_ID_NUM"].isna() |
    df3["DEST_LOC_ID_NUM"].isna()
)

In [ ]:
df3.head(20)

In [9]:
#create next bus and next station columns by shifting
# flags consecutive rides on the same bus service number/same station
df3["same_bus_service"] = (
    df3["BUS_SVC_NUM"].notna() &
    df3["next_BUS_SVC_NUM"].notna() &
    (df3["BUS_SVC_NUM"] == df3["next_BUS_SVC_NUM"])
)
df3["same_station_consecutive"] = (
    df3["DEST_STATION_NAME"].notna() &
    df3["next_orig_station"].notna() &
    (df3["DEST_STATION_NAME"] == df3["next_orig_station"])
)

In [ ]:
df3.head(20)

In [10]:
df3["return_or_intermediate"] = (
    df3["same_bus_service"] |
    df3["same_station_consecutive"]
)

In [12]:
df3["journey_termination_flag"] = (
    df3["is_last_stage"] |
    df3["missing_info"] |
    df3["return_or_intermediate"]
)

In [13]:
df3['journey_termination_flag'].value_counts()


journey_termination_flag
True     2281428
False    1565356
Name: count, dtype: int64

In [ ]:
df3[df3['journey_termination_flag']].head(40)

In [ ]:
df3.head(20)

In [15]:
# Summary counts
print("Unique commuters:", df3["CRD_NUM"].nunique())
print("Last stages:", df3["is_last_stage"].sum())

print("\nMissing info:")
print(df3["missing_info"].value_counts(dropna=False))

print("\nSame bus service:")
print(df3["same_bus_service"].value_counts(dropna=False))

print("\nSame station consecutive:")
print(df3["same_station_consecutive"].value_counts(dropna=False))

print("\nReturn / intermediate:")
print(df3["return_or_intermediate"].value_counts(dropna=False))

print("\nJourney termination flag:")
print(df3["journey_termination_flag"].value_counts(dropna=False))

Unique commuters: 1212990
Last stages: 1212990

Missing info:
missing_info
False    3817059
True       29725
Name: count, dtype: int64

Same bus service:
same_bus_service
False    3521176
True      325608
Name: count, dtype: int64

Same station consecutive:
same_station_consecutive
False    3068298
True      778486
Name: count, dtype: int64

Return / intermediate:
return_or_intermediate
False    2795500
True     1051284
Name: count, dtype: int64

Journey termination flag:
journey_termination_flag
True     2281428
False    1565356
Name: count, dtype: int64


In [16]:
# If you want to see flag rows
display(
    df3.loc[
        df3["journey_termination_flag"],
        [
            "CRD_NUM",
            "ENTRY_TM",
            "EXIT_TM",
            "ORIG_STATION_NAME",
            "DEST_STATION_NAME",
            "next_orig_station",
            "BUS_SVC_NUM",
            "next_BUS_SVC_NUM",
            "is_last_stage",
            "missing_info",
            "same_bus_service",
            "same_station_consecutive",
            "return_or_intermediate",
            "journey_termination_flag"
        ]
    ].head(20)
)

,CRD_NUM,ENTRY_TM,EXIT_TM,ORIG_STATION_NAME,DEST_STATION_NAME,next_orig_station,BUS_SVC_NUM,next_BUS_SVC_NUM,is_last_stage,missing_info,same_bus_service,same_station_consecutive,return_or_intermediate,journey_termination_flag
0,100005879220,2025-02-11 11:20:34,2025-02-11 11:32:25,Admiralty,Yishun,Yishun,NaN,NaN,False,False,False,True,True,True
1,100005879220,2025-02-11 22:32:59,2025-02-11 22:45:03,Yishun,Admiralty,NaN,NaN,NaN,True,False,False,False,False,True
3,100006223599,2025-02-11 07:09:17,2025-02-11 07:18:09,Serangoon NEL,Boon Keng,Boon Keng,NaN,NaN,False,False,False,True,True,True
5,100006223599,2025-02-11 15:10:02,2025-02-11 15:21:55,S'Goon Stn Exit B,Opp Blk 169,NaN,45.0,NaN,True,False,False,False,False,True
6,130013244516,2025-02-11 10:07:29,2025-02-11 10:11:38,Blk 322 CP,Opp One Punggol,One Punggol,382.0,382.0,False,False,True,False,True,True
7,130013244516,2025-02-11 12:15:49,2025-02-11 12:21:50,One Punggol,Opp Blk 322 CP,NaN,382.0,NaN,True,False,False,False,False,True
12,130013244638,2025-02-11 17:05:35,2025-02-11 17:14:58,Opp Bt Gombak Stadium,Blk 111,NaN,871.0,NaN,True,False,False,False,False,True
13,138000455610,2025-02-11 07:19:34,2025-02-11 07:28:26,Bukit Gombak,Yew Tee,Yew Tee,NaN,NaN,False,False,False,True,True,True
14,138000455610,2025-02-11 16:50:34,2025-02-11 17:00:51,Yew Tee,Bukit Gombak,NaN,NaN,NaN,True,False,False,False,False,True
15,138000456325,2025-02-11 09:19:35,2025-02-11 09:25:31,Blk 85,Blk 27,Blk 27,222.0,9.0,False,False,False,True,True,True


In [17]:
# If you want to see transfers
display(
    df3.loc[
        ~df3["journey_termination_flag"],
        [
            "CRD_NUM",
            "ENTRY_TM",
            "EXIT_TM",
            "ORIG_STATION_NAME",
            "DEST_STATION_NAME",
            "next_orig_station",
            "BUS_SVC_NUM",
            "next_BUS_SVC_NUM",
            "is_last_stage",
            "missing_info",
            "same_bus_service",
            "same_station_consecutive",
            "return_or_intermediate",
            "journey_termination_flag"
        ]
    ].head(20)
)

,CRD_NUM,ENTRY_TM,EXIT_TM,ORIG_STATION_NAME,DEST_STATION_NAME,next_orig_station,BUS_SVC_NUM,next_BUS_SVC_NUM,is_last_stage,missing_info,same_bus_service,same_station_consecutive,return_or_intermediate,journey_termination_flag
2,100006223599,2025-02-11 06:56:13,2025-02-11 07:07:33,Blk 172,S'goon Stn Exit A/Blk 413,Serangoon NEL,45.0,NaN,False,False,False,False,False,False
4,100006223599,2025-02-11 14:51:22,2025-02-11 15:03:53,Boon Keng,Serangoon NEL,S'Goon Stn Exit B,NaN,45.0,False,False,False,False,False,False
8,130013244638,2025-02-11 06:40:58,2025-02-11 06:54:37,Tengah CC,Bt Gombak Stadium,Bukit Gombak,871.0,NaN,False,False,False,False,False,False
9,130013244638,2025-02-11 06:58:38,2025-02-11 07:20:00,Bukit Gombak,Woodlands NSEW,Woodlands Int B5,NaN,901.0,False,False,False,False,False,False
10,130013244638,2025-02-11 07:24:42,2025-02-11 07:33:37,Woodlands Int B5,Woodgrove Pr Sch,Woodlands TEL,901.0,NaN,False,False,False,False,False,False
11,130013244638,2025-02-11 16:41:03,2025-02-11 17:01:40,Woodlands TEL,Bukit Gombak,Opp Bt Gombak Stadium,NaN,871.0,False,False,False,False,False,False
16,138000456325,2025-02-11 19:48:36,2025-02-11 19:53:31,Blk 27,Bedok Interchange Alighting 1,Bedok Interchange Boarding Berth 3 TO 10,9.0,17.0,False,False,False,False,False,False
23,138000473167,2025-02-11 08:04:26,2025-02-11 08:18:58,Opp Roxy Sq,Chai Chee Ind Pk,Opp Blk 32,32.0,33.0,False,False,False,False,False,False
25,138000474742,2025-02-11 10:12:00,2025-02-11 10:28:09,Opp White Sands Pr Sch,Pasir Ris Int Alighting B7,Pasir Ris,17.0,NaN,False,False,False,False,False,False
26,138000474742,2025-02-11 10:31:12,2025-02-11 10:46:56,Pasir Ris,Bedok,Bedok Interchange Boarding Berth 3 TO 10,NaN,17.0,False,False,False,False,False,False


In [18]:
# Reasons why flagged.
df3["journey_termination_reason"] = np.select(
    [
        df3["is_last_stage"],
        df3["missing_info"],
        df3["return_or_intermediate"]
    ],
    [
        "last_stage",
        "missing_info",
        "return_or_intermediate"
    ],
    default="candidate_transfer"
)

In [19]:
print(df3["journey_termination_reason"].value_counts(dropna=False))

journey_termination_reason
candidate_transfer        1565356
last_stage                1212990
return_or_intermediate    1047113
missing_info                21325
Name: count, dtype: int64


#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
    - Allowance = Walking speed * walking distance



* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

In [20]:
# Shift next entry time within the same commuter group
df3['next_ENTRY_TM'] = df3.groupby('CRD_NUM')['ENTRY_TM'].shift(-1)

# Time gap = next stage's entry time minus current stage's exit time (in mins)
df3['time_gap_mins'] = (
    df3['next_ENTRY_TM'] - df3['EXIT_TM']
).dt.total_seconds() / 60

# allowance (seconds) = walking distance (m) / walking speed (m/s), then convert to mins
# Rows with no walking distance → allowance is NaN
df3['transfer_allowance_mins'] = (
    df3['walk_distance'] / df3['walking_speed_ms']
) / 60

# True = gap exceeds allowance → new journey (not a transfer)
# If walk_distance is null → allowance is NaN → comparison returns NaN → fillna(True) marks as new journey
df3['exceeds_time_allowance'] = (
    (df3['time_gap_mins'] > df3['transfer_allowance_mins'])
).fillna(True)

print(df3['exceeds_time_allowance'].value_counts())
print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")

print(f"Null walking distance: {df3['walk_distance'].isna().sum():,}")
print(f"Total rows: {len(df3):,}")
print(f"Null %: {df3['walk_distance'].isna().mean() * 100:.2f}%")

exceeds_time_allowance
True     2327847
False    1518937
Name: count, dtype: int64

Total temporal new journey flags: 2,327,847
Null walking distance: 1,239,514
Total rows: 3,846,784
Null %: 32.22%


#### 3. Spatial Criteria (Reasonable walking dist vs Actual walking dist)
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- It flags if the distance between the alighting location of the first journey stage and the boarding location of the next journey stage exceeds a reasonable transfer walking distance.
- Implementation: 
    - Reasonable transfer wwalking distance is the column, walking distance
    - Actual walking distance is time gap between tap out and the next tap in * walking speed
    - To account for time where commuter might be waiting for the next ride instead of walking, we include a 20% buffer.

In [31]:
# actual walking distance (m) = time gap (seconds) × walking speed (m/s)
# time_gap_mins already exists from temporal step, convert back to seconds
df3['actual_walking_dist_m'] = (
    df3['time_gap_mins'] * 60
) * df3['walking_speed_ms']

# Allowance = walk_distance (m) with 20% buffer
# If walk_distance is null → fillna(True) flags as new journey
df3['exceeds_walking_dist'] = (
    df3['actual_walking_dist_m'] > df3['walk_distance'] * 1.2
).fillna(True)

print(df3['exceeds_walking_dist'].value_counts())
print(f"\nTotal spatial new journey flags: {df3['exceeds_walking_dist'].sum():,}")

exceeds_walking_dist
True     2262049
False    1584735
Name: count, dtype: int64

Total spatial new journey flags: 2,262,049


### Combining all 3 criteria

In [48]:
df3['combined_flag'] = (
    df3['journey_termination_flag'] |
    df3['exceeds_time_allowance'] |
    df3['exceeds_walking_dist']
)

## Internal Validity Check with pt_jrny

In [49]:
df5 = pd.read_pickle('../data/df5.pkl')
df5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2777097 entries, 0 to 2777096
Data columns (total 23 columns):
 #   Column              Dtype         
---  ------              -----         
 0   CRD_NUM             int64         
 1   JRNY_DEST_ID_NUM    float64       
 2   JRNY_START_DT       datetime64[ns]
 3   JRNY_START_TM       datetime64[ns]
 4   JRNY_END_DT         datetime64[ns]
 5   JRNY_END_TM         datetime64[ns]
 6   JRNY_ORIG_ID_NUM    float64       
 7   JRNY_DIST_KM_CNT    float64       
 8   JRNY_FARE_AMT       float64       
 9   JRNY_ID_NUM         int64         
 10  JRNY_TM_MIN_CNT     float64       
 11  PATRON_CATG_ID_NUM  int64         
 12  TRNSPT_MODE_CD      int64         
 13  DEST_STATION_NAME   object        
 14  DEST_MRK_ID_NUM     float64       
 15  DEST_LATITUDE       float64       
 16  DEST_LONGITUDE      float64       
 17  DEST_Travel_Type    object        
 18  ORIG_STATION_NAME   object        
 19  ORIG_MRK_ID_NUM     float64       
 20  OR

In [50]:
# same filtering as df3
df5 = df5[df5['PATRON_CATG_ID_NUM'].isin([1, 3, 4])].reset_index(drop=True)

In [51]:
# Merge on CRD_NUM + JRNY_ID_NUM to attach journey boundaries to each ride
df_val = df3.merge(
    df5[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM']],
    on=['CRD_NUM', 'JRNY_ID_NUM'],
    how='inner'
)

# Keep only rides that fall within their journey's time window
df_val = df_val[
    (df_val['ENTRY_TM'] >= df_val['JRNY_START_TM']) &
    (df_val['EXIT_TM'] <= df_val['JRNY_END_TM'])
].reset_index(drop=True)

print(f"Rows after merge + time filter: {len(df_val):,}")

Rows after merge + time filter: 3,669,874


In [52]:
df_val = df_val.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)
# Shift JRNY_ID_NUM within each commuter to get the next ride's journey ID
df_val['next_JRNY_ID_NUM'] = df_val.groupby('CRD_NUM')['JRNY_ID_NUM'].shift(-1)

# true_transfer = True → same journey (actual transfer)
# true_transfer = False → different journey (actual new journey)
df_val['true_transfer'] = (df_val['JRNY_ID_NUM'] == df_val['next_JRNY_ID_NUM'])

# Drop last stage of each commuter — no next ride to compare against
df_val = df_val[df_val['next_JRNY_ID_NUM'].notna()].reset_index(drop=True)

print(df_val['true_transfer'].value_counts())

true_transfer
False    1324143
True     1188828
Name: count, dtype: int64


In [53]:
# helper function
def print_metrics(df, pred_flag_col, label):
    """
    pred_flag_col: True = classifier says NEW JOURNEY, False = classifier says TRANSFER
    true_transfer: True = actual transfer, False = actual new journey
    """
    actual_transfer = df['true_transfer']
    pred_transfer = ~df[pred_flag_col]      # flip: flag=True means new journey, so ~flag = predicted transfer

    tp = (actual_transfer & pred_transfer).sum()     # actual transfer, predicted transfer
    tn = (~actual_transfer & ~pred_transfer).sum()   # actual new journey, predicted new journey
    fp = (~actual_transfer & pred_transfer).sum()    # actual new journey, predicted transfer (merge error)
    fn = (actual_transfer & ~pred_transfer).sum()    # actual transfer, predicted new journey (split error)

    total_actual_transfers = actual_transfer.sum()

    print(f"\n=== {label} ===")
    print(f"TP: {tp:,}  TN: {tn:,}  FP: {fp:,}  FN: {fn:,}")
    print(f"Split rate  (FN / actual transfers): {fn / total_actual_transfers:.4f}")
    print(f"Merge rate  (FP / actual transfers): {fp / total_actual_transfers:.4f}")
    print(f"Sensitivity (TP / (TP+FN)):          {tp / (tp + fn):.4f}")
    print(f"Specificity (TN / (TN+FP)):          {tn / (tn + fp):.4f}")
    print(f"Accuracy    ((TP+TN) / total):       {(tp + tn) / len(df):.4f}")

    print(pd.crosstab(
        actual_transfer,
        pred_transfer,
        rownames=['Actual transfer'],
        colnames=[f'Predicted transfer ({label})']
    ))

In [54]:
print_metrics(df_val, 'journey_termination_flag', 'Binary Criteria')


=== Binary Criteria ===
TP: 959,864  TN: 773,438  FP: 550,705  FN: 228,964
Split rate  (FN / actual transfers): 0.1926
Merge rate  (FP / actual transfers): 0.4632
Sensitivity (TP / (TP+FN)):          0.8074
Specificity (TN / (TN+FP)):          0.5841
Accuracy    ((TP+TN) / total):       0.6897
Predicted transfer (Binary Criteria)   False   True 
Actual transfer                                     
False                                 773438  550705
True                                  228964  959864


In [55]:
print_metrics(df_val, 'exceeds_time_allowance', 'Temporal Criteria')


=== Temporal Criteria ===
TP: 154,624  TN: 1,310,582  FP: 13,561  FN: 1,034,204
Split rate  (FN / actual transfers): 0.8699
Merge rate  (FP / actual transfers): 0.0114
Sensitivity (TP / (TP+FN)):          0.1301
Specificity (TN / (TN+FP)):          0.9898
Accuracy    ((TP+TN) / total):       0.5831
Predicted transfer (Temporal Criteria)    False   True 
Actual transfer                                        
False                                   1310582   13561
True                                    1034204  154624


In [56]:
print_metrics(df_val, 'exceeds_walking_dist', 'Spatial Criteria')


=== Spatial Criteria ===
TP: 217,545  TN: 1,306,771  FP: 17,372  FN: 971,283
Split rate  (FN / actual transfers): 0.8170
Merge rate  (FP / actual transfers): 0.0146
Sensitivity (TP / (TP+FN)):          0.1830
Specificity (TN / (TN+FP)):          0.9869
Accuracy    ((TP+TN) / total):       0.6066
Predicted transfer (Spatial Criteria)    False   True 
Actual transfer                                       
False                                  1306771   17372
True                                    971283  217545


In [57]:
print_metrics(df_val, 'combined_flag', 'Combined Criteria')


=== Combined Criteria ===
TP: 152,855  TN: 1,312,802  FP: 11,341  FN: 1,035,973
Split rate  (FN / actual transfers): 0.8714
Merge rate  (FP / actual transfers): 0.0095
Sensitivity (TP / (TP+FN)):          0.1286
Specificity (TN / (TN+FP)):          0.9914
Accuracy    ((TP+TN) / total):       0.5832
Predicted transfer (Combined Criteria)    False   True 
Actual transfer                                        
False                                   1312802   11341
True                                    1035973  152855


In [58]:
# How much of df_val has null walk_distance?
print("Null walk_distance in df_val:")
print(df_val['walk_distance'].isna().sum(), '/', len(df_val))
print(f"{df_val['walk_distance'].isna().mean()*100:.1f}%")

# Of the true transfers, how many have null walk_distance?
true_transfers = df_val[df_val['true_transfer'] == True]
print("\nNull walk_distance among actual transfers:")
print(true_transfers['walk_distance'].isna().sum(), '/', len(true_transfers))
print(f"{true_transfers['walk_distance'].isna().mean()*100:.1f}%")

Null walk_distance in df_val:
6091 / 2512971
0.2%

Null walk_distance among actual transfers:
2073 / 1188828
0.2%


#ignore below

#### 3. Spatial Criteria (Haversine):
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- Implementation: 
    - Use original clean df to access logtitude and latitude columns
    - Sort the dataset by card number (CRD_NUM) and timestamp (ENTRY_TM) to get consecutive stages for each rider.
    - For each ride, we look at the next stage’s boarding location (latitude and longitude). Rows without a next stage are ignored because there is nothing to compare. The person finished all their trips for the day. 
    - We create a boolean mask to select only rows that have a next stage (NEXT_ENTRY_LAT is not NaN). This helps us to ignore observations without next stages
    - Use Haversine formula to compute the shortest straight-line distance between DEST_LATITUDE & DEST_LONGTITUDE (alighting stop) and the NEXT_ENTRY_LONG & NEXT_ENTRY_LAT (the next entry point). Vectorisation is used because row-by-row computation was too slow. 
    - Allow a maximum transfer distance of 500 metres (we can change this)
    - If the distance exceeds a maximum allowed transfer distance (0.5 km), the row is flagged as spatial_new_journey = True. This indicates that the next ride is likely a new journey.

In [16]:
# Shift boarding coordinates to get "next stage" per rider
df3['NEXT_ENTRY_LAT'] = df3.groupby('CRD_NUM')['ORIG_LATITUDE'].shift(-1)
df3['NEXT_ENTRY_LON'] = df3.groupby('CRD_NUM')['ORIG_LONGITUDE'].shift(-1)

In [17]:
import numpy as np
#use vectorized function so it runs much faster than row by row
def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371  # Earth radius in km
    return c * r

In [18]:
#compute distance to next stage if it exists
mask = df3['NEXT_ENTRY_LAT'].notna()  # skip last stage (ie. if there is no next tap in) 
df3.loc[mask, 'distance_to_next_km'] = haversine_vectorized(
    df3.loc[mask, 'DEST_LATITUDE'].values,
    df3.loc[mask, 'DEST_LONGITUDE'].values,
    df3.loc[mask, 'NEXT_ENTRY_LAT'].values,
    df3.loc[mask, 'NEXT_ENTRY_LON'].values
)

In [19]:
#flag spatial transfers
# set max reasonable walking transfer distance = 0.5 km 
# we can change this threshold
max_transfer_distance_km = 0.5

#initialise as False. Only mark as True if distance to next stage exceeds threshold
df3['spatial_new_journey'] = False
df3.loc[mask, 'spatial_new_journey'] = df3.loc[mask, 'distance_to_next_km'] > max_transfer_distance_km

In [20]:
df3['spatial_new_journey'].value_counts()

spatial_new_journey
False    3657236
True      189548
Name: count, dtype: int64

# Internal Validity Check
- Drop rows with missing critical data
- Combine `pt_ride` and `pt_jrny` and only keep those where the ride entry time >= journey start time and ride exit time <= journey end time
- Generate confusion matrix calculate metrics (split rate, merge rate, sensitivity, specificity, accuracy)

In [ ]:
df5 = pd.read_pickle('../data/df5.pkl')

In [ ]:
df5.info()

In [ ]:
# Drop rows with missing critical data
critical_cols = [
    'JRNY_START_TM', 'JRNY_END_TM',
    'JRNY_ORIG_ID_NUM', 'JRNY_DEST_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df5)
df5 = df5.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df5)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

In [ ]:
df4 = df5[df5['JRNY_START_DT'] == '2025-02-11']

#### 1. Binary Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df6 = df2.merge(df4, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df6 = df6[(df6['ENTRY_DT'] >= df6['JRNY_START_DT']) & (df6['EXIT_DT'] <= df6['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df6['next_JRNY_ID_NUM'] = df6['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df6['true_same_journey'] = (df6['JRNY_ID_NUM'] == df6['next_JRNY_ID_NUM'])

print(df6['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_same_journey'],
    df6['same_service_return'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true = (df6['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 2. Temporal Criteria

In [ ]:
df6['true_new_journey'] = (df6['JRNY_ID_NUM'] != df6['next_JRNY_ID_NUM'])

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_new_journey'],
    df6['temporal_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
# Split Error Rate (FN)
split_error = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 3. Spatial Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df7 = df3.merge(df5, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df7 = df7[(df7['ENTRY_DT'] >= df7['JRNY_START_DT']) & (df7['EXIT_DT'] <= df7['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df7['next_JRNY_ID_NUM'] = df7['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df7['true_same_journey'] = (df7['JRNY_ID_NUM'] == df7['next_JRNY_ID_NUM'])

print(df7['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df7['true_new_journey'],
    df7['spatial_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true2 = (df7['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == False)).sum()
split_rate = split_error / total_true2
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == True)).sum()
merge_rate = merge_error / total_true2
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true2 + merge_error + tn)
print("Accuracy:", accuracy)